## Read data from bronze

In [0]:
from pyspark.sql import functions as F, Window
from pyspark.sql.functions import col
from pyspark.sql.types import (
    StringType, DoubleType, StructType, StructField,
    ArrayType, IntegerType
)
from delta.tables import DeltaTable

BRONZE_TABLE = "tabular.dataexpert.smc_stocks_bronze"

bronze_df = spark.table(BRONZE_TABLE)
#Writing to two silver tables
#Metrics - these are for ticker and trade date
SILVER_METRICS_TABLE = "tabular.dataexpert.smc_stocks_silver_metrics"
#Pairs - these are for ticker_a, ticker_b and trade_Date
SILVER_PAIRS_TABLE   = "tabular.dataexpert.smc_stocks_silver_pairs"

## Daily OHLCV Aggregation
Roll up minute bars into daily summaries

In [0]:
daily_df = (
    bronze_df
    .groupBy("ticker", "trade_date")
    .agg(
        F.sort_array(F.collect_list(F.struct("timestamp", "open", "close"))).alias("ts_ohlc"),
        F.max("high").alias("day_high"),
        F.min("low").alias("day_low"),
        F.sum("volume").alias("day_volume"),
        F.count("*").alias("bar_count"),
    )
)

daily_df = (
    daily_df
    .withColumn("day_open", F.col("ts_ohlc")[0]["open"])
    .withColumn("day_close", F.col("ts_ohlc")[F.size("ts_ohlc") - 1]["close"])
    .withColumn("all_closes", F.transform(F.col("ts_ohlc"), lambda x: x["close"]))
    .drop("ts_ohlc")
)

daily_count = daily_df.count()
print(f"Daily rows: {daily_count:,}")

daily_df.show(5, truncate=False)

## Technical Indicators - RSI & Volatility
Calculate RSI (Relative Strength Index) and a custom volatility score for each ticker-day. RSI is a momentum oscillator that measures the speed and magnitude of price changes on a 0-100 scale.

In [0]:
@udf(returnType=DoubleType())
def compute_rsi(closes_array, period=14):
    """Calculate RSI from an array of intraday close prices."""
    if closes_array is None or len(closes_array) < period + 1:
        return None
    gains = []
    losses = []
    for i in range(1, len(closes_array)):
        diff = closes_array[i] - closes_array[i - 1]
        if diff > 0:
            gains.append(diff)
            losses.append(0.0)
        else:
            gains.append(0.0)
            losses.append(abs(diff))
    if len(gains) < period:
        return None
    avg_gain = sum(gains[:period]) / period
    avg_loss = sum(losses[:period]) / period
    for i in range(period, len(gains)):
        avg_gain = (avg_gain * (period - 1) + gains[i]) / period
        avg_loss = (avg_loss * (period - 1) + losses[i]) / period
    if avg_loss == 0:
        return 100.0
    rs = avg_gain / avg_loss
    return 100.0 - (100.0 / (1.0 + rs))

avg_volume_df = daily_df.groupBy("ticker").agg(
    F.avg("day_volume").alias("avg_volume")
)

# Volatility - How much does this stock wiggle during the day, scaled to a yearly number?
@udf(returnType=DoubleType())
def compute_volatility(closes_array):
    """Intraday volatility: std dev of minute returns, annualized."""
    if closes_array is None or len(closes_array) < 2: # <2 as we need at least 2 prices to compute a return
        return None
    returns = [(closes_array[i] - closes_array[i-1]) / closes_array[i-1]
               for i in range(1, len(closes_array))
               if closes_array[i-1] != 0]
    if len(returns) < 2:
        return None
    mean = sum(returns) / len(returns)
    variance = sum((r - mean) ** 2 for r in returns) / (len(returns) - 1) #How far do returns deviate from the average
    return float(variance ** 0.5 * (252 * 390) ** 0.5)  # annualized

daily_with_avg = daily_df.join(avg_volume_df, on="ticker", how="left")
                               
daily_indicators = (
    daily_with_avg
    .withColumn("rsi", compute_rsi(F.col("all_closes")))
    .withColumn("volatility_score", compute_volatility(F.col("all_closes")))
    .withColumn(
        "signal_regime",
            F.when(F.col("rsi").isNull() | F.col("day_volume").isNull() | F.col("avg_volume").isNull(), "UNKNOWN")
            .when(F.col("avg_volume") == 0, "UNKNOWN")
            .when((F.col("rsi") > 70) & ((F.col("day_volume") / F.col("avg_volume")) > 1.5), "STRONG_SELL")
            .when(F.col("rsi") > 70, "SELL")
            .when((F.col("rsi") < 30) & ((F.col("day_volume")/F.col("avg_volume")) > 1.5), "STRONG_BUY")
            .when(F.col("rsi") < 30, "BUY")
            .when(F.col("rsi").between(45, 55), "NEUTRAL")
            .when(F.col("rsi") > 55, "LEAN_SELL")
            .otherwise("LEAN_BUY")                          
         )
    )

daily_indicators = daily_indicators.drop("all_closes", "avg_volume")

daily_indicators.select(
    "ticker", "trade_date", "day_close", "rsi", "signal_regime", "volatility_score"
).show(20, truncate=False)

indicator_count = daily_indicators.count()
print(f"Rows with indicators: {indicator_count:,}")

## Cross-Ticker Correlation Pairs
For every pair of tickers on the same trading day, compute the price return similarity

In [0]:
minute_returns = bronze_df.select(
    "ticker", "trade_date", "timestamp",
    ((F.col("close") - F.col("open")) / F.col("open")).alias("minute_return")
)

left = minute_returns.select(
    F.col("ticker").alias("ticker_a"),
    F.col("trade_date"),
    F.col("timestamp"),
    F.col("minute_return").alias("ret_a"),
).alias("a")

right = minute_returns.select(
    F.col("ticker").alias("ticker_b"),
    F.col("trade_date"),
    F.col("timestamp"),
    F.col("minute_return").alias("ret_b"),
).alias("b")

ticker_pairs = left.join(
    right,
    (F.col("a.trade_date") == F.col("b.trade_date")) &
    (F.col("a.timestamp") == F.col("b.timestamp")) &
    (F.col("a.ticker_a") < F.col("b.ticker_b"))
)

correlated = ticker_pairs.groupBy("ticker_a", "ticker_b", F.col("a.trade_date").alias("trade_date")).agg(
    F.corr("ret_a", "ret_b").alias("intraday_corr"),
    F.avg(F.abs(F.col("ret_a") - F.col("ret_b"))).alias("return_diff")
).withColumn("silver_ingestion_timestamp", F.current_timestamp())

print("Correlated Data")
correlated.orderBy(F.desc("trade_date"), "ticker_a", "ticker_b").show(10, truncate=False)

## Daily Signal Report
Combine indicators with the previous day's signal (using a lag join) to detect regime changes

In [0]:
window = Window.partitionBy("ticker").orderBy("trade_date")

signal_report = daily_indicators.select(
    "ticker"
    ,"trade_date"
    ,"day_open"
    ,"day_high"
    ,"day_low"
    ,"day_volume"
    ,"bar_count"
    ,"signal_regime"
    , "rsi"
    ,"volatility_score"
    , "day_close"
    ,F.lag("signal_regime").over(window).alias("prev_regime")
    ,F.lag("rsi").over(window).alias("prev_rsi")
    ,F.lag("day_close").over(window).alias("prev_close")
)

signal_report = signal_report.withColumn(
    "regime_changed",
    F.when(
        F.col("signal_regime") != F.col("prev_regime"), True
    ).otherwise(False),
).withColumn(
    "day_return_pct",
    F.round(
        (F.col("day_close") - F.col("prev_close")) / F.col("prev_close") * 100,
        2,
    ),
).withColumn("silver_ingestion_timestamp", F.current_timestamp())

regime_changes = signal_report.filter(F.col("regime_changed") == True)
print(f"Regime changes detected: {regime_changes.count():,}")

signal_report.orderBy("trade_date", "ticker").show(10, truncate=False)

## Write data to silver tables

In [0]:
def merge_to_silver(source_df, target_table, merge_keys:list):
    if spark.catalog.tableExists(target_table):
        delta_table = DeltaTable.forName(spark, target_table)
        merge_condition = " AND ".join([f"target.{key} = source.{key}" for key in merge_keys])
        (
            delta_table.alias("target")
            .merge(
                source_df.alias("source"),
                merge_condition,
            )
            .whenMatchedUpdateAll()
            .whenNotMatchedInsertAll()
            .execute()
        )
    else:
        source_df.write.saveAsTable(target_table)
        print(f"Written: {target_table} | Rows: {source_df.count():,}")

merge_to_silver(signal_report, SILVER_METRICS_TABLE, ["ticker", "trade_date"])
merge_to_silver(correlated, SILVER_PAIRS_TABLE, ["ticker_a", "ticker_b", "trade_date"])